# 16 GHSL Six-City Population and Degree-of-Urbanisation Model Training

This notebook trains and evaluates the two missing predictive targets using only the six established training cities:

1. **GHS-POP:** predict next-epoch `population_total` using Linear Regression and Random Forest Regression.
2. **GHS-SMOD / Degree of Urbanisation:** predict next-epoch `smod_class` using Random Forest multiclass classification.

The chronological design matches the built-surface experiment:

- Training: target epochs 1980–2010, 205,800 examples
- Validation: 2010→2015, 29,400 examples
- Test: 2015→2020, 29,400 examples

Persistence baselines are mandatory:

- Population persistence: next population = current population
- SMOD persistence: next SMOD class = current SMOD class

Jaipur is not read, scored, or used in this notebook. A separate notebook will later load the frozen built-surface, population, and SMOD models and evaluate all three on Jaipur.

All Spark and HDFS work runs through `/content/drive/MyDrive/remote_exec.sh`. Run one cell at a time.

In [1]:
# ============================================================
# CELL 1: BOOTSTRAP
# ============================================================
from google.colab import drive
from pathlib import Path
import base64, csv, json, os, shutil, subprocess, time

drive.mount('/content/drive',force_remount=False)
D=Path('/content/drive/MyDrive');REMOTE_EXEC=D/'remote_exec.sh';KEY=D/'hadoop-hdfs.pem';LOCAL_KEY=Path('/content/hadoop-hdfs.pem')
ART=D/'GHSL_Urban_Growth_Model'/'six_city_population_smod_models_v1';ART.mkdir(parents=True,exist_ok=True)
TRAINING_MANIFEST=D/'GHSL_Urban_Growth_Model'/'model_training_v1'/'training_manifest.json'
for p in [REMOTE_EXEC,KEY,TRAINING_MANIFEST]:
    if not p.is_file(): raise FileNotFoundError(str(p))
REMOTE_EXEC.chmod(0o700)
if LOCAL_KEY.exists(): LOCAL_KEY.unlink()
shutil.copy2(KEY,LOCAL_KEY);LOCAL_KEY.chmod(0o600)
def run_remote(command,timeout=300,show_output=True,raise_on_error=True):
    r=subprocess.run(['bash',str(REMOTE_EXEC),command],text=True,capture_output=True,timeout=timeout)
    if show_output and r.stdout: print(r.stdout)
    if show_output and r.stderr: print('REMOTE STDERR:');print(r.stderr)
    if raise_on_error and r.returncode!=0: raise RuntimeError('Remote command failed with return code '+str(r.returncode))
    return r
MANIFEST12=json.loads(TRAINING_MANIFEST.read_text(encoding='utf-8'))
BUILT_FEATURES=MANIFEST12['results']['features']
LONGITUDINAL='/data/ghsl/modeling/six_city_longitudinal_v1'
TRANSITIONS='/data/ghsl/modeling/six_city_population_smod_transitions_v1'
MODEL_ROOT='/data/ghsl/modeling/models/six_city_population_smod_v1'
POP_LR=MODEL_ROOT+'/population_linear_regression';POP_RF=MODEL_ROOT+'/population_random_forest';SMOD_RF=MODEL_ROOT+'/smod_random_forest'
EVAL_ROOT='/data/ghsl/modeling/evaluation/six_city_population_smod_v1'
print('Built-surface feature contract loaded:',len(BUILT_FEATURES))
print('Jaipur included: NO')
print('CELL 1: SUCCESS')

Mounted at /content/drive
Built-surface feature contract loaded: 31
Jaipur included: NO
CELL 1: SUCCESS


In [2]:
# ============================================================
# CELL 2: REMOTE/HDFS/PYSPARK ML PREFLIGHT
# ============================================================
cmd=r"""set -e
source /home/ubuntu/ghsl-sedona-env.sh
export HADOOP_HOME=/usr/local/hadoop;export PATH="${HADOOP_HOME}/bin:${HADOOP_HOME}/sbin:${PATH}";export SETUPTOOLS_USE_DISTUTILS=local
for d in NameNode DataNode SecondaryNameNode;do jps|awk '{print $2}'|grep -qx "$d";done
hdfs dfsadmin -safemode get|grep -qi 'Safe mode is OFF'
hdfs dfs -test -e /data/ghsl/modeling/six_city_longitudinal_v1
python3 - <<'PY_REMOTE'
import setuptools,distutils
from pyspark.ml import Pipeline
from pyspark.ml.feature import Imputer,VectorAssembler,StringIndexer,IndexToString
from pyspark.ml.regression import LinearRegression,RandomForestRegressor
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import RegressionEvaluator,MulticlassClassificationEvaluator
print('ML imports: PASS')
PY_REMOTE
echo SIX_CITY_POPULATION_SMOD_PREFLIGHT_SUCCESS
"""
r=run_remote(cmd,300)
print('CELL 2: SUCCESS')

ML imports: PASS
SIX_CITY_POPULATION_SMOD_PREFLIGHT_SUCCESS

REMOTE STDERR:

CELL 2: SUCCESS


In [3]:
# ============================================================
# CELL 3: BUILD THREE-TARGET SIX-CITY TRANSITIONS
# Adds next built surface, population, and SMOD labels.
# ============================================================
payload=base64.b64encode(json.dumps({'input':LONGITUDINAL,'output':TRANSITIONS}).encode()).decode()
cmd=r"""set -e
source /home/ubuntu/ghsl-sedona-env.sh
export HADOOP_HOME=/usr/local/hadoop;export PATH="${HADOOP_HOME}/bin:${HADOOP_HOME}/sbin:${PATH}"
P=/tmp/nb16_trans.json;S=/tmp/nb16_trans.py;echo __P__|base64 -d > "$P"
cat > "$S" <<'PY_REMOTE'
from pyspark.sql import SparkSession,functions as F,Window
import json,sys
p=json.load(open(sys.argv[1]));s=SparkSession.builder.appName('Six-City-Population-SMOD-Transitions').getOrCreate();s.sparkContext.setLogLevel('WARN')
try:
 fs=s._jvm.org.apache.hadoop.fs.FileSystem.get(s._jsc.hadoopConfiguration());op=s._jvm.org.apache.hadoop.fs.Path(p['output'])
 if fs.exists(op):raise RuntimeError('Refusing overwrite: '+p['output'])
 d=s.read.parquet(p['input']);w=Window.partitionBy('city_slug','grid_cell_id').orderBy('epoch')
 t=(d.withColumn('to_epoch',F.lead('epoch').over(w)).withColumn('next_built_surface_total_m2',F.lead(F.col('built_surface_total_m2').cast('double')).over(w)).withColumn('next_population_total',F.lead(F.col('population_total').cast('double')).over(w)).withColumn('next_smod_class',F.lead(F.col('smod_class')).over(w)).withColumnRenamed('epoch','from_epoch').filter((F.col('to_epoch').isNotNull())&((F.col('to_epoch')-F.col('from_epoch'))==5)))
 rows=t.count();keys=t.select('city_slug','grid_cell_id','from_epoch','to_epoch').distinct().count();nulls=t.filter(F.col('next_population_total').isNull()|F.isnan('next_population_total')|F.col('next_smod_class').isNull()).count()
 split={int(r['to_epoch']):r['count'] for r in t.groupBy('to_epoch').count().collect()}
 if rows!=264600 or keys!=264600 or nulls!=0 or len(split)!=9 or any(v!=29400 for v in split.values()):raise RuntimeError(repr({'rows':rows,'keys':keys,'nulls':nulls,'split':split}))
 t.repartition('to_epoch','city_slug').write.mode('errorifexists').partitionBy('to_epoch','city_slug').parquet(p['output'])
 print('TRANSITION_JSON_BEGIN');print(json.dumps({'rows':rows,'keys':keys,'split':split},sort_keys=True));print('TRANSITION_JSON_END');print('SIX_CITY_POPULATION_SMOD_TRANSITIONS_SUCCESS')
finally:s.stop()
PY_REMOTE
spark-submit --master local[2] --driver-memory 5g --conf spark.ui.enabled=false "$S" "$P";rm -f "$S" "$P"
""".replace('__P__',payload)
r=run_remote(cmd,7200,show_output=False)
if 'SIX_CITY_POPULATION_SMOD_TRANSITIONS_SUCCESS' not in r.stdout:
    print(r.stdout[-12000:]);print(r.stderr[-6000:]);raise RuntimeError('Transition build failed')
TRANSITION_SUMMARY=json.loads(r.stdout.split('TRANSITION_JSON_BEGIN',1)[1].split('TRANSITION_JSON_END',1)[0].strip())
print(json.dumps(TRANSITION_SUMMARY,indent=2));print('CELL 3: SUCCESS')

{
  "keys": 264600,
  "rows": 264600,
  "split": {
    "1980": 29400,
    "1985": 29400,
    "1990": 29400,
    "1995": 29400,
    "2000": 29400,
    "2005": 29400,
    "2010": 29400,
    "2015": 29400,
    "2020": 29400
  }
}
CELL 3: SUCCESS


In [4]:
# ============================================================
# CELL 4: FREEZE TARGET-SPECIFIC FEATURE CONTRACTS
# Future-state and derived change fields are prohibited.
# ============================================================
future_or_leakage={'next_built_surface_total_m2','next_population_total','next_smod_class','to_epoch','built_surface_growth_m2','population_change','label'}
# Reuse the proven 31 current-state numeric features. They include current
# population and current SMOD, which are legitimate autoregressive inputs.
POPULATION_FEATURES=[x for x in BUILT_FEATURES if x not in future_or_leakage]
SMOD_FEATURES=[x for x in BUILT_FEATURES if x not in future_or_leakage]
if len(POPULATION_FEATURES)!=31 or len(SMOD_FEATURES)!=31:raise RuntimeError('Feature-contract count changed unexpectedly')
contracts={'population':{'target':'next_population_total','features':POPULATION_FEATURES,'problem':'regression'},'smod':{'target':'next_smod_class','features':SMOD_FEATURES,'problem':'multiclass_classification'}}
(ART/'feature_contracts.json').write_text(json.dumps(contracts,indent=2,sort_keys=True)+'\n')
print(json.dumps(contracts,indent=2));print('CELL 4: SUCCESS')

{
  "population": {
    "target": "next_population_total",
    "features": [
      "grid_1km_col",
      "grid_1km_row",
      "grid_min_x",
      "grid_max_x",
      "grid_min_y",
      "grid_max_y",
      "valid_100m_pixels",
      "population_coverage_fraction",
      "population_total",
      "population_mean_100m",
      "population_min_100m",
      "population_max_100m",
      "zero_population_pixels",
      "positive_population_pixels",
      "smod_class",
      "built_valid_100m_pixels",
      "built_observed_grid_area_m2",
      "built_coverage_fraction",
      "built_surface_nres_m2",
      "built_surface_residential_m2",
      "built_surface_fraction",
      "built_surface_percent",
      "built_surface_nres_fraction",
      "built_surface_residential_fraction",
      "nres_share_of_built_surface",
      "built_100m_pixels",
      "nres_100m_pixels",
      "population_per_built_hectare",
      "population_per_residential_built_hectare",
      "grid_resolution_m",
      "from

In [5]:
# ============================================================
# CELL 5: TRAIN CANDIDATE MODELS ON TARGET YEARS THROUGH 2010
# ============================================================
payload=base64.b64encode(json.dumps({'input':TRANSITIONS,'pop_lr':POP_LR,'pop_rf':POP_RF,'smod_rf':SMOD_RF,'pop_features':POPULATION_FEATURES,'smod_features':SMOD_FEATURES}).encode()).decode()
cmd=r"""set -e
source /home/ubuntu/ghsl-sedona-env.sh
export HADOOP_HOME=/usr/local/hadoop;export PATH="${HADOOP_HOME}/bin:${HADOOP_HOME}/sbin:${PATH}";export SETUPTOOLS_USE_DISTUTILS=local
P=/tmp/nb16_train.json;S=/tmp/nb16_train.py;echo __P__|base64 -d > "$P"
cat > "$S" <<'PY_REMOTE'
from pyspark.sql import SparkSession,functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import Imputer,VectorAssembler,StringIndexer,IndexToString
from pyspark.ml.regression import LinearRegression,RandomForestRegressor
from pyspark.ml.classification import RandomForestClassifier
import json,sys
p=json.load(open(sys.argv[1]));s=SparkSession.builder.appName('Six-City-Population-SMOD-Training').getOrCreate();s.sparkContext.setLogLevel('WARN')
try:
 fs=s._jvm.org.apache.hadoop.fs.FileSystem.get(s._jsc.hadoopConfiguration())
 for x in [p['pop_lr'],p['pop_rf'],p['smod_rf']]:
  if fs.exists(s._jvm.org.apache.hadoop.fs.Path(x)):raise RuntimeError('Refusing overwrite: '+x)
 d=s.read.parquet(p['input']);train=d.filter(F.col('to_epoch')<=2010)
 if train.count()!=205800:raise RuntimeError('Expected 205800 training rows')
 def prep(features,prefix):
  imp=[prefix+x for x in features];return [Imputer(inputCols=features,outputCols=imp,strategy='median'),VectorAssembler(inputCols=imp,outputCol='features',handleInvalid='keep')]
 pp=prep(p['pop_features'],'pop_imp_')
 lr=Pipeline(stages=pp+[LinearRegression(featuresCol='features',labelCol='next_population_total',predictionCol='population_prediction',maxIter=100,regParam=0.01,elasticNetParam=0.0)]).fit(train);lr.write().save(p['pop_lr'])
 rf=Pipeline(stages=pp+[RandomForestRegressor(featuresCol='features',labelCol='next_population_total',predictionCol='population_prediction',numTrees=80,maxDepth=12,minInstancesPerNode=5,seed=42,subsamplingRate=0.8)]).fit(train);rf.write().save(p['pop_rf'])
 sp=prep(p['smod_features'],'smod_imp_');indexer=StringIndexer(inputCol='next_smod_class',outputCol='smod_label',handleInvalid='error').fit(train)
 clf=RandomForestClassifier(featuresCol='features',labelCol='smod_label',predictionCol='smod_prediction_index',probabilityCol='smod_probability',numTrees=100,maxDepth=12,minInstancesPerNode=5,seed=42,subsamplingRate=0.8)
 sm=Pipeline(stages=[indexer]+sp+[clf,IndexToString(inputCol='smod_prediction_index',outputCol='smod_prediction',labels=indexer.labels)]).fit(train);sm.write().save(p['smod_rf'])
 print('SIX_CITY_POPULATION_SMOD_MODELS_TRAINED')
finally:s.stop()
PY_REMOTE
spark-submit --master local[2] --driver-memory 6g --conf spark.ui.enabled=false --conf spark.sql.shuffle.partitions=24 --conf spark.driver.maxResultSize=1g "$S" "$P";rm -f "$S" "$P"
""".replace('__P__',payload)
r=run_remote(cmd,10800,show_output=False)
if 'SIX_CITY_POPULATION_SMOD_MODELS_TRAINED' not in r.stdout:
    print(r.stdout[-16000:]);print(r.stderr[-8000:]);raise RuntimeError('Training failed')
print('CELL 5: SUCCESS')

CELL 5: SUCCESS


In [10]:
# ============================================================
# CORRECTED CELL 6:
# EVALUATE POPULATION AND DEGREE-OF-URBANISATION MODELS
#
# Evaluation design:
#   - Validation: 2010 -> 2015
#   - Test:       2015 -> 2020
#
# Population methods:
#   - Linear Regression
#   - Random Forest Regression
#   - Population persistence baseline
#
# Degree-of-Urbanisation methods:
#   - Random Forest Classification
#   - SMOD persistence baseline
#
# Critical fix:
#   Each Spark ML pipeline transforms the original evaluation
#   DataFrame independently. Only prediction columns are joined.
#
# Safety:
#   - Does not retrain models
#   - Does not modify the transition dataset
#   - Refuses to overwrite an existing evaluation root
# ============================================================

for required_name in [
    "TRANSITIONS",
    "POP_LR",
    "POP_RF",
    "SMOD_RF",
    "EVAL_ROOT",
    "run_remote",
]:
    if required_name not in globals():
        raise RuntimeError(
            required_name
            + " is unavailable. Run Cells 1-5 first."
        )


payload = {
    "input": TRANSITIONS,
    "pop_lr": POP_LR,
    "pop_rf": POP_RF,
    "smod_rf": SMOD_RF,
    "out": EVAL_ROOT,
}


payload_b64 = base64.b64encode(
    json.dumps(
        payload,
        sort_keys=True,
    ).encode(
        "utf-8"
    )
).decode(
    "ascii"
)


command = r'''
set -e

source /home/ubuntu/ghsl-sedona-env.sh

export HADOOP_HOME=/usr/local/hadoop
export PATH="${HADOOP_HOME}/bin:${HADOOP_HOME}/sbin:${PATH}"
export SETUPTOOLS_USE_DISTUTILS=local

PAYLOAD="/tmp/nb16_eval_corrected_v2.json"
SCRIPT="/tmp/nb16_eval_corrected_v2.py"

echo "__PAYLOAD__" \
| base64 -d \
> "$PAYLOAD"


cat > "$SCRIPT" <<'PY_REMOTE'
from pyspark.sql import (
    SparkSession,
    functions as F,
)

from pyspark.ml import (
    PipelineModel,
)

from pyspark.ml.evaluation import (
    RegressionEvaluator,
)

import json
import sys


# ============================================================
# LOAD PAYLOAD
# ============================================================

payload_path = sys.argv[1]


with open(
    payload_path,
    "r",
    encoding="utf-8",
) as handle:
    payload = json.load(
        handle
    )


input_path = payload[
    "input"
]

population_lr_path = payload[
    "pop_lr"
]

population_rf_path = payload[
    "pop_rf"
]

smod_rf_path = payload[
    "smod_rf"
]

output_root = payload[
    "out"
]


EXPECTED_TOTAL_ROWS = 58800
EXPECTED_ROWS_PER_EPOCH = 29400
EXPECTED_ROWS_PER_CITY_EPOCH = 4900
EXPECTED_CITY_COUNT = 6
EXPECTED_METRIC_RECORDS = 14


# ============================================================
# START SPARK
# ============================================================

spark = (
    SparkSession.builder
    .appName(
        "Six-City-Population-SMOD-Evaluation-Corrected-V2"
    )
    .getOrCreate()
)

spark.sparkContext.setLogLevel(
    "WARN"
)


try:
    # ========================================================
    # REFUSE TO OVERWRITE AN EXISTING EVALUATION ROOT
    # ========================================================

    filesystem = (
        spark._jvm
        .org.apache.hadoop.fs.FileSystem
        .get(
            spark._jsc.hadoopConfiguration()
        )
    )

    evaluation_output_path = (
        spark._jvm
        .org.apache.hadoop.fs.Path(
            output_root
        )
    )

    if filesystem.exists(
        evaluation_output_path
    ):
        raise RuntimeError(
            "Refusing to overwrite existing evaluation root: "
            + output_root
        )


    # ========================================================
    # LOAD THE ORIGINAL VALIDATION AND TEST ROWS
    # ========================================================

    evaluation = (
        spark.read
        .parquet(
            input_path
        )
        .filter(
            F.col(
                "to_epoch"
            ).isin(
                2015,
                2020,
            )
        )
    )


    key_columns = [
        "city_slug",
        "grid_cell_id",
        "from_epoch",
        "to_epoch",
    ]


    required_columns = (
        key_columns
        + [
            "population_total",
            "next_population_total",
            "smod_class",
            "next_smod_class",
        ]
    )


    missing_columns = [
        column_name
        for column_name
        in required_columns
        if column_name
        not in evaluation.columns
    ]


    if missing_columns:
        raise RuntimeError(
            "Evaluation dataset is missing required columns: "
            + repr(
                missing_columns
            )
        )


    evaluation_rows = (
        evaluation.count()
    )


    evaluation_keys = (
        evaluation
        .select(
            *key_columns
        )
        .distinct()
        .count()
    )


    if evaluation_rows != EXPECTED_TOTAL_ROWS:
        raise RuntimeError(
            "Expected 58,800 evaluation rows, found "
            + str(
                evaluation_rows
            )
        )


    if evaluation_keys != EXPECTED_TOTAL_ROWS:
        raise RuntimeError(
            "Expected 58,800 unique evaluation keys, found "
            + str(
                evaluation_keys
            )
        )


    null_target_rows = (
        evaluation
        .filter(
            F.col(
                "next_population_total"
            ).isNull()
            | F.isnan(
                "next_population_total"
            )
            | F.col(
                "next_smod_class"
            ).isNull()
        )
        .count()
    )


    if null_target_rows != 0:
        raise RuntimeError(
            "Found "
            + str(
                null_target_rows
            )
            + " rows with invalid evaluation targets."
        )


    # ========================================================
    # LOAD THE THREE TRAINED PIPELINES
    # ========================================================

    population_lr_model = (
        PipelineModel.load(
            population_lr_path
        )
    )


    population_rf_model = (
        PipelineModel.load(
            population_rf_path
        )
    )


    smod_rf_model = (
        PipelineModel.load(
            smod_rf_path
        )
    )


    # ========================================================
    # TRANSFORM THE ORIGINAL DATAFRAME INDEPENDENTLY
    #
    # Do not apply one pipeline to another pipeline's output.
    # Every pipeline creates its own imputed columns and its own
    # "features" column, so they must run independently.
    # ========================================================

    population_lr_predictions = (
        population_lr_model
        .transform(
            evaluation
        )
        .select(
            *key_columns,
            F.col(
                "population_prediction"
            )
            .cast(
                "double"
            )
            .alias(
                "lr_population_prediction"
            ),
        )
    )


    population_rf_predictions = (
        population_rf_model
        .transform(
            evaluation
        )
        .select(
            *key_columns,
            F.col(
                "population_prediction"
            )
            .cast(
                "double"
            )
            .alias(
                "rf_population_prediction"
            ),
        )
    )


    smod_rf_predictions = (
        smod_rf_model
        .transform(
            evaluation
        )
        .select(
            *key_columns,
            F.col(
                "smod_prediction"
            )
            .cast(
                "string"
            )
            .alias(
                "smod_prediction_string"
            ),
            F.col(
                "smod_prediction_index"
            )
            .cast(
                "double"
            )
            .alias(
                "smod_prediction_index"
            ),
        )
    )


    # ========================================================
    # VALIDATE EACH INDEPENDENT MODEL OUTPUT
    # ========================================================

    independent_outputs = [
        (
            "population_linear_regression",
            population_lr_predictions,
        ),
        (
            "population_random_forest",
            population_rf_predictions,
        ),
        (
            "smod_random_forest",
            smod_rf_predictions,
        ),
    ]


    for (
        model_name,
        prediction_frame,
    ) in independent_outputs:
        prediction_rows = (
            prediction_frame.count()
        )

        prediction_keys = (
            prediction_frame
            .select(
                *key_columns
            )
            .distinct()
            .count()
        )

        if prediction_rows != EXPECTED_TOTAL_ROWS:
            raise RuntimeError(
                model_name
                + ": expected 58,800 prediction rows, found "
                + str(
                    prediction_rows
                )
            )

        if prediction_keys != EXPECTED_TOTAL_ROWS:
            raise RuntimeError(
                model_name
                + ": expected 58,800 unique prediction keys, found "
                + str(
                    prediction_keys
                )
            )


    # ========================================================
    # CREATE A CLEAN BASE DATAFRAME
    #
    # Intermediate columns created by earlier failed or manual
    # transformations are excluded defensively.
    # ========================================================

    excluded_intermediate_columns = {
        "features",
        "population_prediction",
        "lr_population_prediction",
        "rf_population_prediction",
        "smod_prediction",
        "smod_prediction_string",
        "smod_prediction_index",
        "smod_probability",
        "rawPrediction",
        "probability",
        "prediction",
    }


    base_columns = [
        column_name
        for column_name
        in evaluation.columns
        if (
            column_name
            not in excluded_intermediate_columns
            and not column_name.startswith(
                "pop_imp_"
            )
            and not column_name.startswith(
                "smod_imp_"
            )
        )
    ]


    base_evaluation = (
        evaluation.select(
            *base_columns
        )
    )


    # ========================================================
    # JOIN ONLY THE FINAL PREDICTION COLUMNS
    # ========================================================

    combined = (
        base_evaluation

        .join(
            population_lr_predictions,
            on=key_columns,
            how="inner",
        )

        .join(
            population_rf_predictions,
            on=key_columns,
            how="inner",
        )

        .join(
            smod_rf_predictions,
            on=key_columns,
            how="inner",
        )

        .withColumn(
            "population_persistence",
            F.col(
                "population_total"
            ).cast(
                "double"
            ),
        )

        .withColumn(
            "smod_persistence",
            F.col(
                "smod_class"
            ).cast(
                "string"
            ),
        )

        .withColumn(
            "smod_actual",
            F.col(
                "next_smod_class"
            ).cast(
                "string"
            ),
        )

        .withColumn(
            "lr_population_error",
            (
                F.col(
                    "lr_population_prediction"
                )
                - F.col(
                    "next_population_total"
                ).cast(
                    "double"
                )
            ),
        )

        .withColumn(
            "rf_population_error",
            (
                F.col(
                    "rf_population_prediction"
                )
                - F.col(
                    "next_population_total"
                ).cast(
                    "double"
                )
            ),
        )

        .withColumn(
            "persistence_population_error",
            (
                F.col(
                    "population_persistence"
                )
                - F.col(
                    "next_population_total"
                ).cast(
                    "double"
                )
            ),
        )

        .withColumn(
            "lr_population_abs_error",
            F.abs(
                F.col(
                    "lr_population_error"
                )
            ),
        )

        .withColumn(
            "rf_population_abs_error",
            F.abs(
                F.col(
                    "rf_population_error"
                )
            ),
        )

        .withColumn(
            "persistence_population_abs_error",
            F.abs(
                F.col(
                    "persistence_population_error"
                )
            ),
        )

        .withColumn(
            "actual_population_change",
            (
                F.col(
                    "next_population_total"
                ).cast(
                    "double"
                )
                - F.col(
                    "population_total"
                ).cast(
                    "double"
                )
            ),
        )

        .withColumn(
            "lr_predicted_population_change",
            (
                F.col(
                    "lr_population_prediction"
                )
                - F.col(
                    "population_total"
                ).cast(
                    "double"
                )
            ),
        )

        .withColumn(
            "rf_predicted_population_change",
            (
                F.col(
                    "rf_population_prediction"
                )
                - F.col(
                    "population_total"
                ).cast(
                    "double"
                )
            ),
        )
    )


    # ========================================================
    # VALIDATE THE JOINED DATAFRAME
    # ========================================================

    combined_rows = (
        combined.count()
    )


    combined_keys = (
        combined
        .select(
            *key_columns
        )
        .distinct()
        .count()
    )


    invalid_prediction_rows = (
        combined
        .filter(
            F.col(
                "lr_population_prediction"
            ).isNull()
            | F.isnan(
                "lr_population_prediction"
            )
            | F.col(
                "rf_population_prediction"
            ).isNull()
            | F.isnan(
                "rf_population_prediction"
            )
            | F.col(
                "smod_prediction_string"
            ).isNull()
        )
        .count()
    )


    if combined_rows != EXPECTED_TOTAL_ROWS:
        raise RuntimeError(
            "The prediction joins changed the row count. "
            "Expected 58,800 rows, found "
            + str(
                combined_rows
            )
        )


    if combined_keys != EXPECTED_TOTAL_ROWS:
        raise RuntimeError(
            "The joined prediction keys are not unique. "
            "Expected 58,800 keys, found "
            + str(
                combined_keys
            )
        )


    if invalid_prediction_rows != 0:
        raise RuntimeError(
            "Found "
            + str(
                invalid_prediction_rows
            )
            + " rows with null or invalid predictions."
        )


    # ========================================================
    # CHECK TARGET-EPOCH AND CITY COUNTS
    # ========================================================

    epoch_counts = {
        int(
            row[
                "to_epoch"
            ]
        ): int(
            row[
                "count"
            ]
        )
        for row
        in combined
        .groupBy(
            "to_epoch"
        )
        .count()
        .collect()
    }


    expected_epoch_counts = {
        2015: EXPECTED_ROWS_PER_EPOCH,
        2020: EXPECTED_ROWS_PER_EPOCH,
    }


    if epoch_counts != expected_epoch_counts:
        raise RuntimeError(
            "Unexpected target-epoch counts: "
            + repr(
                epoch_counts
            )
        )


    city_epoch_counts = (
        combined
        .groupBy(
            "city_slug",
            "to_epoch",
        )
        .count()
    )


    invalid_city_epoch_counts = (
        city_epoch_counts
        .filter(
            F.col(
                "count"
            )
            != EXPECTED_ROWS_PER_CITY_EPOCH
        )
        .count()
    )


    city_slugs = sorted(
        row[
            0
        ]
        for row
        in combined
        .select(
            "city_slug"
        )
        .distinct()
        .collect()
    )


    if len(city_slugs) != EXPECTED_CITY_COUNT:
        raise RuntimeError(
            "Expected six evaluated cities, found "
            + repr(
                city_slugs
            )
        )


    if invalid_city_epoch_counts != 0:
        raise RuntimeError(
            "One or more city-target-epoch groups "
            "do not contain exactly 4,900 rows."
        )


    # ========================================================
    # WRITE CELL-LEVEL PREDICTIONS
    # ========================================================

    cell_prediction_output = (
        output_root
        + "/cell_predictions"
    )


    (
        combined
        .repartition(
            "to_epoch",
            "city_slug",
        )
        .write
        .mode(
            "errorifexists"
        )
        .partitionBy(
            "to_epoch",
            "city_slug",
        )
        .parquet(
            cell_prediction_output
        )
    )


    # ========================================================
    # METRIC HELPERS
    # ========================================================

    def regression_metrics(
        dataframe,
        prediction_column,
    ):
        evaluation_frame = (
            dataframe
            .withColumn(
                "evaluation_prediction",
                F.col(
                    prediction_column
                ).cast(
                    "double"
                ),
            )
        )

        return {
            metric_name: (
                RegressionEvaluator(
                    labelCol=(
                        "next_population_total"
                    ),
                    predictionCol=(
                        "evaluation_prediction"
                    ),
                    metricName=metric_name,
                )
                .evaluate(
                    evaluation_frame
                )
            )
            for metric_name
            in [
                "mae",
                "rmse",
                "r2",
            ]
        }


    def population_bias(
        dataframe,
        prediction_column,
    ):
        return float(
            dataframe
            .agg(
                F.avg(
                    (
                        F.col(
                            prediction_column
                        )
                        - F.col(
                            "next_population_total"
                        ).cast(
                            "double"
                        )
                    )
                ).alias(
                    "bias"
                )
            )
            .first()[
                "bias"
            ]
        )


    def exact_accuracy(
        dataframe,
        prediction_column,
    ):
        row_count = (
            dataframe.count()
        )

        if row_count <= 0:
            raise RuntimeError(
                "Cannot calculate accuracy on an empty DataFrame."
            )

        correct_count = (
            dataframe
            .filter(
                F.col(
                    prediction_column
                )
                == F.col(
                    "smod_actual"
                )
            )
            .count()
        )

        return float(
            correct_count
            / row_count
        )


    # ========================================================
    # OVERALL AND CITY-WISE METRICS
    # ========================================================

    metric_records = []


    for target_epoch in [
        2015,
        2020,
    ]:
        epoch_frame = (
            combined
            .filter(
                F.col(
                    "to_epoch"
                )
                == target_epoch
            )
        )


        scopes = [
            (
                "overall",
                "ALL",
                epoch_frame,
            )
        ]


        for city_slug in city_slugs:
            scopes.append(
                (
                    "city",
                    city_slug,
                    epoch_frame.filter(
                        F.col(
                            "city_slug"
                        )
                        == city_slug
                    ),
                )
            )


        for (
            scope_name,
            city_slug,
            scope_frame,
        ) in scopes:
            scope_rows = (
                scope_frame.count()
            )

            expected_scope_rows = (
                EXPECTED_ROWS_PER_EPOCH
                if scope_name
                == "overall"
                else EXPECTED_ROWS_PER_CITY_EPOCH
            )


            if scope_rows != expected_scope_rows:
                raise RuntimeError(
                    "Unexpected row count for "
                    + scope_name
                    + " "
                    + city_slug
                    + " "
                    + str(
                        target_epoch
                    )
                    + ": "
                    + str(
                        scope_rows
                    )
                )


            metric_records.append(
                {
                    "to_epoch": (
                        target_epoch
                    ),
                    "scope": (
                        scope_name
                    ),
                    "city_slug": (
                        city_slug
                    ),
                    "rows": (
                        scope_rows
                    ),
                    "population_lr": (
                        regression_metrics(
                            scope_frame,
                            "lr_population_prediction",
                        )
                    ),
                    "population_rf": (
                        regression_metrics(
                            scope_frame,
                            "rf_population_prediction",
                        )
                    ),
                    "population_persistence": (
                        regression_metrics(
                            scope_frame,
                            "population_persistence",
                        )
                    ),
                    "population_lr_bias": (
                        population_bias(
                            scope_frame,
                            "lr_population_prediction",
                        )
                    ),
                    "population_rf_bias": (
                        population_bias(
                            scope_frame,
                            "rf_population_prediction",
                        )
                    ),
                    "population_persistence_bias": (
                        population_bias(
                            scope_frame,
                            "population_persistence",
                        )
                    ),
                    "smod_rf_accuracy": (
                        exact_accuracy(
                            scope_frame,
                            "smod_prediction_string",
                        )
                    ),
                    "smod_persistence_accuracy": (
                        exact_accuracy(
                            scope_frame,
                            "smod_persistence",
                        )
                    ),
                }
            )


    if len(
        metric_records
    ) != EXPECTED_METRIC_RECORDS:
        raise RuntimeError(
            "Expected 14 metric records, found "
            + str(
                len(
                    metric_records
                )
            )
        )


    # ========================================================
    # SMOD CONFUSION TABLES
    # ========================================================

    confusion_records = [
        row.asDict(
            recursive=True
        )
        for row
        in combined
        .groupBy(
            "to_epoch",
            "smod_actual",
            "smod_prediction_string",
        )
        .count()
        .orderBy(
            "to_epoch",
            "smod_actual",
            "smod_prediction_string",
        )
        .collect()
    ]


    persistence_confusion_records = [
        row.asDict(
            recursive=True
        )
        for row
        in combined
        .groupBy(
            "to_epoch",
            "smod_actual",
            "smod_persistence",
        )
        .count()
        .orderBy(
            "to_epoch",
            "smod_actual",
            "smod_persistence",
        )
        .collect()
    ]


    if not confusion_records:
        raise RuntimeError(
            "The SMOD confusion table is empty."
        )


    if not persistence_confusion_records:
        raise RuntimeError(
            "The SMOD persistence confusion table is empty."
        )


    # ========================================================
    # FLATTEN AND WRITE METRIC TABLES
    # ========================================================

    flattened_metric_records = []


    for record in metric_records:
        flattened_metric_records.append(
            {
                "to_epoch": (
                    int(
                        record[
                            "to_epoch"
                        ]
                    )
                ),
                "scope": (
                    record[
                        "scope"
                    ]
                ),
                "city_slug": (
                    record[
                        "city_slug"
                    ]
                ),
                "rows": (
                    int(
                        record[
                            "rows"
                        ]
                    )
                ),
                "population_lr_mae": (
                    float(
                        record[
                            "population_lr"
                        ][
                            "mae"
                        ]
                    )
                ),
                "population_lr_rmse": (
                    float(
                        record[
                            "population_lr"
                        ][
                            "rmse"
                        ]
                    )
                ),
                "population_lr_r2": (
                    float(
                        record[
                            "population_lr"
                        ][
                            "r2"
                        ]
                    )
                ),
                "population_rf_mae": (
                    float(
                        record[
                            "population_rf"
                        ][
                            "mae"
                        ]
                    )
                ),
                "population_rf_rmse": (
                    float(
                        record[
                            "population_rf"
                        ][
                            "rmse"
                        ]
                    )
                ),
                "population_rf_r2": (
                    float(
                        record[
                            "population_rf"
                        ][
                            "r2"
                        ]
                    )
                ),
                "population_persistence_mae": (
                    float(
                        record[
                            "population_persistence"
                        ][
                            "mae"
                        ]
                    )
                ),
                "population_persistence_rmse": (
                    float(
                        record[
                            "population_persistence"
                        ][
                            "rmse"
                        ]
                    )
                ),
                "population_persistence_r2": (
                    float(
                        record[
                            "population_persistence"
                        ][
                            "r2"
                        ]
                    )
                ),
                "population_lr_bias": (
                    float(
                        record[
                            "population_lr_bias"
                        ]
                    )
                ),
                "population_rf_bias": (
                    float(
                        record[
                            "population_rf_bias"
                        ]
                    )
                ),
                "population_persistence_bias": (
                    float(
                        record[
                            "population_persistence_bias"
                        ]
                    )
                ),
                "smod_rf_accuracy": (
                    float(
                        record[
                            "smod_rf_accuracy"
                        ]
                    )
                ),
                "smod_persistence_accuracy": (
                    float(
                        record[
                            "smod_persistence_accuracy"
                        ]
                    )
                ),
            }
        )


    metrics_output = (
        output_root
        + "/metric_table"
    )

    smod_confusion_output = (
        output_root
        + "/smod_confusion"
    )

    smod_persistence_confusion_output = (
        output_root
        + "/smod_persistence_confusion"
    )


    (
        spark.createDataFrame(
            flattened_metric_records
        )
        .write
        .mode(
            "errorifexists"
        )
        .parquet(
            metrics_output
        )
    )


    (
        spark.createDataFrame(
            confusion_records
        )
        .write
        .mode(
            "errorifexists"
        )
        .parquet(
            smod_confusion_output
        )
    )


    (
        spark.createDataFrame(
            persistence_confusion_records
        )
        .write
        .mode(
            "errorifexists"
        )
        .parquet(
            smod_persistence_confusion_output
        )
    )


    # ========================================================
    # POST-WRITE VALIDATION
    # ========================================================

    written_prediction_rows = (
        spark.read
        .parquet(
            cell_prediction_output
        )
        .count()
    )


    written_metric_rows = (
        spark.read
        .parquet(
            metrics_output
        )
        .count()
    )


    written_confusion_rows = (
        spark.read
        .parquet(
            smod_confusion_output
        )
        .count()
    )


    written_persistence_confusion_rows = (
        spark.read
        .parquet(
            smod_persistence_confusion_output
        )
        .count()
    )


    if written_prediction_rows != EXPECTED_TOTAL_ROWS:
        raise RuntimeError(
            "Post-write prediction count failed. "
            "Expected 58,800, found "
            + str(
                written_prediction_rows
            )
        )


    if written_metric_rows != EXPECTED_METRIC_RECORDS:
        raise RuntimeError(
            "Post-write metric count failed. "
            "Expected 14, found "
            + str(
                written_metric_rows
            )
        )


    # ========================================================
    # FINAL RESULT
    # ========================================================

    output = {
        "metrics": (
            metric_records
        ),
        "confusion": (
            confusion_records
        ),
        "persistence_confusion": (
            persistence_confusion_records
        ),
        "prediction_rows": (
            written_prediction_rows
        ),
        "metric_rows": (
            written_metric_rows
        ),
        "confusion_rows": (
            written_confusion_rows
        ),
        "persistence_confusion_rows": (
            written_persistence_confusion_rows
        ),
        "city_slugs": (
            city_slugs
        ),
        "epoch_counts": (
            epoch_counts
        ),
        "evaluation_root": (
            output_root
        ),
    }


    print(
        "EVAL_JSON_BEGIN"
    )


    print(
        json.dumps(
            output,
            sort_keys=True,
        )
    )


    print(
        "EVAL_JSON_END"
    )


    print(
        "SIX_CITY_POPULATION_SMOD_EVALUATION_SUCCESS"
    )


finally:
    spark.stop()
PY_REMOTE


# ============================================================
# COMPILE THE GENERATED SCRIPT BEFORE SPARK-SUBMIT
# ============================================================

python3 -m py_compile \
    "$SCRIPT"


spark-submit \
    --master local[2] \
    --driver-memory 6g \
    --conf spark.ui.enabled=false \
    --conf spark.sql.shuffle.partitions=24 \
    --conf spark.driver.maxResultSize=1g \
    "$SCRIPT" \
    "$PAYLOAD"


rm -f \
    "$SCRIPT" \
    "$PAYLOAD"


echo
echo "Temporary evaluation files removed: PASS"
echo "Source datasets modified: NO"
echo "CORRECTED_CELL_6_V2_REMOTE_SUCCESS"
'''


command = (
    command.replace(
        "__PAYLOAD__",
        payload_b64,
    )
)


# ============================================================
# RUN REMOTE EVALUATION
# ============================================================

evaluation_result = run_remote(
    command,
    timeout=10800,
    show_output=True,
)


if (
    "SIX_CITY_POPULATION_SMOD_EVALUATION_SUCCESS"
    not in evaluation_result.stdout
):
    raise RuntimeError(
        "The corrected evaluation did not "
        "produce its Spark success marker."
    )


if (
    "CORRECTED_CELL_6_V2_REMOTE_SUCCESS"
    not in evaluation_result.stdout
):
    raise RuntimeError(
        "The corrected evaluation did not "
        "produce its remote completion marker."
    )


# ============================================================
# PARSE THE EVALUATION JSON
# ============================================================

begin_marker = (
    "EVAL_JSON_BEGIN"
)

end_marker = (
    "EVAL_JSON_END"
)


begin_index = (
    evaluation_result.stdout.find(
        begin_marker
    )
)


end_index = (
    evaluation_result.stdout.find(
        end_marker
    )
)


if begin_index < 0:
    raise RuntimeError(
        "Evaluation JSON begin marker was not found."
    )


if end_index < 0:
    raise RuntimeError(
        "Evaluation JSON end marker was not found."
    )


if end_index <= begin_index:
    raise RuntimeError(
        "Evaluation JSON markers are in the wrong order."
    )


json_start = (
    begin_index
    + len(
        begin_marker
    )
)


evaluation_json = (
    evaluation_result.stdout[
        json_start:
        end_index
    ]
    .strip()
)


if not evaluation_json:
    raise RuntimeError(
        "The evaluation JSON payload is empty."
    )


EVALUATION = json.loads(
    evaluation_json
)


# ============================================================
# LOCAL VALIDATION
# ============================================================

if (
    EVALUATION.get(
        "prediction_rows"
    )
    != 58800
):
    raise RuntimeError(
        "Expected 58,800 written evaluation rows, found "
        + str(
            EVALUATION.get(
                "prediction_rows"
            )
        )
    )


if (
    EVALUATION.get(
        "metric_rows"
    )
    != 14
):
    raise RuntimeError(
        "Expected 14 metric records, found "
        + str(
            EVALUATION.get(
                "metric_rows"
            )
        )
    )


if (
    len(
        EVALUATION.get(
            "city_slugs",
            [],
        )
    )
    != 6
):
    raise RuntimeError(
        "Expected results for six cities, found "
        + repr(
            EVALUATION.get(
                "city_slugs"
            )
        )
    )


expected_epoch_counts = {
    "2015": 29400,
    "2020": 29400,
}


actual_epoch_counts = {
    str(
        key
    ): value
    for (
        key,
        value,
    ) in EVALUATION.get(
        "epoch_counts",
        {},
    ).items()
}


if actual_epoch_counts != expected_epoch_counts:
    raise RuntimeError(
        "Unexpected target-epoch counts: "
        + repr(
            actual_epoch_counts
        )
    )


# ============================================================
# FINAL REPORT
# ============================================================

print(
    "Prediction rows:",
    EVALUATION[
        "prediction_rows"
    ],
)


print(
    "Metric records:",
    len(
        EVALUATION[
            "metrics"
        ]
    ),
)


print(
    "SMOD confusion records:",
    len(
        EVALUATION[
            "confusion"
        ]
    ),
)


print(
    "SMOD persistence confusion records:",
    len(
        EVALUATION[
            "persistence_confusion"
        ]
    ),
)


print(
    "Cities:",
    EVALUATION[
        "city_slugs"
    ],
)


print(
    "Target epochs:",
    sorted(
        int(
            key
        )
        for key
        in EVALUATION[
            "epoch_counts"
        ]
    ),
)


print(
    "Population Linear Regression evaluated: YES"
)


print(
    "Population Random Forest evaluated: YES"
)


print(
    "Population persistence evaluated: YES"
)


print(
    "SMOD Random Forest evaluated: YES"
)


print(
    "SMOD persistence evaluated: YES"
)


print(
    "Source datasets modified: NO"
)


print(
    "CELL 6: SUCCESS"
)


EVAL_JSON_BEGIN
{"city_slugs": ["bengaluru", "chennai_metro", "delhi_ncr", "hyderabad", "mumbai_metro", "pune_metro"], "confusion": [{"count": 4455, "smod_actual": "0", "smod_prediction_string": "0", "to_epoch": 2015}, {"count": 124, "smod_actual": "10", "smod_prediction_string": "10", "to_epoch": 2015}, {"count": 1, "smod_actual": "11", "smod_prediction_string": "10", "to_epoch": 2015}, {"count": 6334, "smod_actual": "11", "smod_prediction_string": "11", "to_epoch": 2015}, {"count": 184, "smod_actual": "11", "smod_prediction_string": "12", "to_epoch": 2015}, {"count": 1, "smod_actual": "11", "smod_prediction_string": "30", "to_epoch": 2015}, {"count": 610, "smod_actual": "12", "smod_prediction_string": "11", "to_epoch": 2015}, {"count": 5929, "smod_actual": "12", "smod_prediction_string": "12", "to_epoch": 2015}, {"count": 29, "smod_actual": "12", "smod_prediction_string": "13", "to_epoch": 2015}, {"count": 89, "smod_actual": "12", "smod_prediction_string": "21", "to_epoch": 2015}, {"

In [7]:
# ============================================================
# CELL 6 FAILURE DIAGNOSTIC
#
# Read-only:
#   - Does not rerun evaluation
#   - Does not delete HDFS outputs
#   - Does not modify trained models
# ============================================================

diagnostic_command = r'''
source /home/ubuntu/ghsl-sedona-env.sh

export HADOOP_HOME=/usr/local/hadoop
export PATH="${HADOOP_HOME}/bin:${HADOOP_HOME}/sbin:${PATH}"
export SETUPTOOLS_USE_DISTUTILS=local

EVALUATION_ROOT="/data/ghsl/modeling/evaluation/six_city_population_smod_v1"
MODEL_ROOT="/data/ghsl/modeling/models/six_city_population_smod_v1"

echo "============================================================"
echo "CELL 6 POPULATION/SMOD EVALUATION DIAGNOSTIC"
echo "============================================================"

echo
echo "========== PRESERVED TEMPORARY FILES =========="

for file_path in \
    /tmp/nb16_eval.json \
    /tmp/nb16_eval.py
do
    if [ -f "$file_path" ]; then
        echo "FOUND: $file_path"
        ls -lh "$file_path"
    else
        echo "MISSING: $file_path"
    fi
done

echo
echo "========== TRAINED MODEL INVENTORY =========="

for model_name in \
    population_linear_regression \
    population_random_forest \
    smod_random_forest
do
    model_path="$MODEL_ROOT/$model_name"

    echo
    echo "Model: $model_name"

    if hdfs dfs -test -e "$model_path"; then
        echo "STATUS: PRESENT"

        hdfs dfs -count -h \
            "$model_path" \
            || true

        if hdfs dfs -test -e \
            "$model_path/metadata/_SUCCESS"
        then
            echo "METADATA_SUCCESS: PRESENT"
        else
            echo "METADATA_SUCCESS: MISSING"
        fi
    else
        echo "STATUS: MISSING"
    fi
done

echo
echo "========== EVALUATION ROOT =========="

if hdfs dfs -test -e "$EVALUATION_ROOT"; then
    echo "EVALUATION_ROOT_EXISTS: YES"

    hdfs dfs -ls -R \
        "$EVALUATION_ROOT" \
        | head -300 \
        || true
else
    echo "EVALUATION_ROOT_EXISTS: NO"
fi

echo
echo "========== TRANSITION DATASET =========="

TRANSITION_PATH="/data/ghsl/modeling/six_city_population_smod_transitions_v1"

if hdfs dfs -test -e "$TRANSITION_PATH"; then
    echo "TRANSITION_DATASET: PRESENT"

    hdfs dfs -count -h \
        "$TRANSITION_PATH"
else
    echo "TRANSITION_DATASET: MISSING"
fi

echo
echo "========== PYSPARK ML IMPORT TEST =========="

python3 - <<'PY_REMOTE'
import setuptools
import distutils

from pyspark.ml import PipelineModel

from pyspark.ml.evaluation import (
    RegressionEvaluator,
    MulticlassClassificationEvaluator,
)

print(
    "PySpark ML imports: PASS"
)
PY_REMOTE

echo
echo "========== DISK AND MEMORY STATUS =========="

df -h / /tmp

free -h

echo
echo "========== RECENT KERNEL MEMORY EVENTS =========="

dmesg 2>/dev/null \
    | grep -Ei \
        "out of memory|oom|killed process" \
    | tail -20 \
    || true

echo
echo "CELL_6_DIAGNOSTIC_COMPLETE"
'''

diagnostic_result = run_remote(
    diagnostic_command,
    timeout=600,
    show_output=True,
)

print(
    "Return code:",
    diagnostic_result.returncode,
)

print(
    "CELL 6 DIAGNOSTIC: SUCCESS"
)

CELL 6 POPULATION/SMOD EVALUATION DIAGNOSTIC

========== PRESERVED TEMPORARY FILES ==========
FOUND: /tmp/nb16_eval.json
-rw-rw-r-- 1 ubuntu ubuntu 419 Nov 15 08:47 /tmp/nb16_eval.json
FOUND: /tmp/nb16_eval.py
-rw-rw-r-- 1 ubuntu ubuntu 2.7K Nov 15 08:47 /tmp/nb16_eval.py

========== TRAINED MODEL INVENTORY ==========

Model: population_linear_regression
STATUS: PRESENT
          11           12             15.5 K /data/ghsl/modeling/models/six_city_population_smod_v1/population_linear_regression
METADATA_SUCCESS: PRESENT

Model: population_random_forest
STATUS: PRESENT
          12           14             12.3 M /data/ghsl/modeling/models/six_city_population_smod_v1/population_random_forest
METADATA_SUCCESS: PRESENT

Model: smod_random_forest
STATUS: PRESENT
          17           20              3.7 M /data/ghsl/modeling/models/six_city_population_smod_v1/smod_random_forest
METADATA_SUCCESS: PRESENT

========== EVALUATION ROOT ==========
EVALUATION_ROOT_EXISTS: NO

========== TRANSI

In [8]:
# ============================================================
# CELL 6A: VISIBLE RETRY OF THE PRESERVED EVALUATION
#
# Run only when:
#   EVALUATION_ROOT_EXISTS: NO
#
# Reuses the exact Cell 6 script and payload.
# ============================================================

visible_evaluation_command = r'''
set -e

source /home/ubuntu/ghsl-sedona-env.sh

export HADOOP_HOME=/usr/local/hadoop
export PATH="${HADOOP_HOME}/bin:${HADOOP_HOME}/sbin:${PATH}"
export SETUPTOOLS_USE_DISTUTILS=local

SCRIPT="/tmp/nb16_eval.py"
PAYLOAD="/tmp/nb16_eval.json"
EVALUATION_ROOT="/data/ghsl/modeling/evaluation/six_city_population_smod_v1"

echo "============================================================"
echo "VISIBLE POPULATION/SMOD EVALUATION RETRY"
echo "============================================================"

if [ ! -f "$SCRIPT" ]; then
    echo "ERROR: Preserved evaluation script is missing."
    exit 20
fi

if [ ! -f "$PAYLOAD" ]; then
    echo "ERROR: Preserved evaluation payload is missing."
    exit 21
fi

if hdfs dfs -test -e "$EVALUATION_ROOT"; then
    echo "ERROR: Evaluation root already exists:"
    echo "$EVALUATION_ROOT"
    echo "Nothing was deleted."
    exit 22
fi

echo
echo "Evaluation script:"
ls -lh "$SCRIPT"

echo
echo "Evaluation payload:"
cat "$PAYLOAD"

echo
echo "========== STARTING VISIBLE SPARK EVALUATION =========="

spark-submit \
    --master local[2] \
    --driver-memory 6g \
    --conf spark.ui.enabled=false \
    --conf spark.sql.shuffle.partitions=24 \
    --conf spark.driver.maxResultSize=1g \
    "$SCRIPT" \
    "$PAYLOAD"
'''

visible_evaluation_result = subprocess.run(
    [
        "bash",
        str(REMOTE_EXEC),
        visible_evaluation_command,
    ],
    text=True,
    capture_output=True,
    timeout=10800,
)

print(
    "REMOTE RETURN CODE:",
    visible_evaluation_result.returncode,
)

print()
print(
    "================ REMOTE STDOUT ================"
)
print(
    visible_evaluation_result.stdout
)

print()
print(
    "================ REMOTE STDERR ================"
)
print(
    visible_evaluation_result.stderr
)

if visible_evaluation_result.returncode == 0:
    print(
        "VISIBLE EVALUATION RETRY: SUCCESS"
    )
else:
    print(
        "VISIBLE EVALUATION RETRY: FAILED"
    )

REMOTE RETURN CODE: 1

================ REMOTE STDOUT ================
VISIBLE POPULATION/SMOD EVALUATION RETRY

Evaluation script:
-rw-rw-r-- 1 ubuntu ubuntu 2.7K Nov 15 08:47 /tmp/nb16_eval.py

Evaluation payload:
{"input": "/data/ghsl/modeling/six_city_population_smod_transitions_v1", "pop_lr": "/data/ghsl/modeling/models/six_city_population_smod_v1/population_linear_regression", "pop_rf": "/data/ghsl/modeling/models/six_city_population_smod_v1/population_random_forest", "smod_rf": "/data/ghsl/modeling/models/six_city_population_smod_v1/smod_random_forest", "out": "/data/ghsl/modeling/evaluation/six_city_population_smod_v1"}
========== STARTING VISIBLE SPARK EVALUATION ==========
Traceback (most recent call last):
  File "/tmp/nb16_eval.py", line 11, in <module>
    both=PipelineModel.load(p['pop_rf']).transform(lr).withColumnRenamed('population_prediction','rf_population_prediction')
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/ubuntu/ghsl-sedona-env/lib/pyt

In [11]:
# ============================================================
# CELL 7: DISPLAY MODEL COMPARISONS AND SELECT BY VALIDATION
# ============================================================
import pandas as pd
records=[]
for x in EVALUATION['metrics']:
 if x['scope']=='overall':
  for model,key in [('Population Linear Regression','population_lr'),('Population Random Forest','population_rf'),('Population persistence','population_persistence')]:records.append({'to_epoch':x['to_epoch'],'model':model,**x[key]})
  records += [{'to_epoch':x['to_epoch'],'model':'SMOD Random Forest','accuracy':x['smod_rf_accuracy']},{'to_epoch':x['to_epoch'],'model':'SMOD persistence','accuracy':x['smod_persistence_accuracy']}]
OVERALL=pd.DataFrame(records)
print('OVERALL POPULATION METRICS');display(OVERALL[OVERALL.model.str.startswith('Population')].sort_values(['to_epoch','rmse']).style.format({'mae':'{:,.2f}','rmse':'{:,.2f}','r2':'{:.6f}'}))
print('OVERALL SMOD ACCURACY');display(OVERALL[OVERALL.model.str.startswith('SMOD')].sort_values(['to_epoch','accuracy'],ascending=[True,False]).style.format({'accuracy':'{:.2%}'}))
val=OVERALL[(OVERALL.to_epoch==2015)&OVERALL.model.str.startswith('Population')];SELECTED_POPULATION_MODEL=val.loc[val.rmse.idxmin(),'model']
print('Population model selected by 2015 validation RMSE:',SELECTED_POPULATION_MODEL)
print('SMOD learned model candidate: SMOD Random Forest')
print('CELL 7: SUCCESS')

OVERALL POPULATION METRICS


,to_epoch,model,mae,r2,rmse,accuracy
0,2015,Population Linear Regression,288.33,0.993877,713.86,nan
2,2015,Population persistence,303.51,0.988959,958.64,nan
1,2015,Population Random Forest,356.48,0.978413,"1,340.43",nan
5,2020,Population Linear Regression,305.70,0.995755,644.63,nan
7,2020,Population persistence,316.83,0.990332,972.82,nan
6,2020,Population Random Forest,450.48,0.971140,"1,680.81",nan


OVERALL SMOD ACCURACY


,to_epoch,model,mae,r2,rmse,accuracy
3,2015,SMOD Random Forest,nan,nan,nan,91.92%
4,2015,SMOD persistence,nan,nan,nan,91.35%
8,2020,SMOD Random Forest,nan,nan,nan,89.38%
9,2020,SMOD persistence,nan,nan,nan,88.90%


Population model selected by 2015 validation RMSE: Population Linear Regression
SMOD learned model candidate: SMOD Random Forest
CELL 7: SUCCESS


In [12]:
# ============================================================
# CELL 8: EXPORT REPORT-READY METRICS, CONFUSION TABLE, FIGURES
# ============================================================
metric_df=pd.json_normalize(EVALUATION['metrics']);confusion_df=pd.DataFrame(EVALUATION['confusion'])
metric_df.to_csv(ART/'population_smod_metrics.csv',index=False);confusion_df.to_csv(ART/'smod_confusion_2015_2020.csv',index=False)
(ART/'population_smod_evaluation.json').write_text(json.dumps(EVALUATION,indent=2,sort_keys=True)+'\n')
import matplotlib.pyplot as plt
FIG=ART/'figures';FIG.mkdir(exist_ok=True)
pop=OVERALL[OVERALL.model.str.startswith('Population')].pivot(index='model',columns='to_epoch',values='rmse');pop.plot(kind='bar',figsize=(10,5),title='Six-city population RMSE');plt.xticks(rotation=15,ha='right');plt.tight_layout();plt.savefig(FIG/'population_rmse.png',dpi=180);plt.close()
sm=OVERALL[OVERALL.model.str.startswith('SMOD')].pivot(index='model',columns='to_epoch',values='accuracy');sm.plot(kind='bar',figsize=(9,5),title='Six-city Degree of Urbanisation accuracy');plt.ylim(0,1);plt.xticks(rotation=0);plt.tight_layout();plt.savefig(FIG/'smod_accuracy.png',dpi=180);plt.close()
print('Saved metrics and figures:',ART);print('CELL 8: SUCCESS')

Saved metrics and figures: /content/drive/MyDrive/GHSL_Urban_Growth_Model/six_city_population_smod_models_v1
CELL 8: SUCCESS


In [13]:
# ============================================================
# CELL 9: FINAL MODEL MANIFEST AND COMPLETION GATE
# ============================================================
manifest={'created_at_utc':time.strftime('%Y-%m-%dT%H:%M:%SZ',time.gmtime()),'training_cities':['bengaluru','hyderabad','pune_metro','chennai_metro','mumbai_metro','delhi_ncr'],'jaipur_used':False,'training_target_epochs':'1980-2010','validation':'2010->2015','test':'2015->2020','population_models':{'linear_regression':POP_LR,'random_forest':POP_RF,'selected_by_validation':SELECTED_POPULATION_MODEL},'smod_model':{'random_forest':SMOD_RF,'benchmark':'current-class persistence'},'feature_contracts':str(ART/'feature_contracts.json'),'evaluation_root':EVAL_ROOT,'metrics_file':str(ART/'population_smod_metrics.csv')}
(ART/'six_city_population_smod_model_manifest.json').write_text(json.dumps(manifest,indent=2,sort_keys=True)+'\n')
print('Population model training: COMPLETE')
print('Degree of Urbanisation model training: COMPLETE')
print('2015 validation: COMPLETE')
print('2020 test: COMPLETE')
print('Persistence baselines: COMPLETE')
print('Jaipur used: NO')
print('NOTEBOOK_16_SIX_CITY_POPULATION_SMOD_TRAINING_SUCCESS')
print('CELL 9: SUCCESS')

Population model training: COMPLETE
Degree of Urbanisation model training: COMPLETE
2015 validation: COMPLETE
2020 test: COMPLETE
Persistence baselines: COMPLETE
Jaipur used: NO
NOTEBOOK_16_SIX_CITY_POPULATION_SMOD_TRAINING_SUCCESS
CELL 9: SUCCESS


## Next notebook

After reviewing these 2015 and 2020 results, freeze the chosen population model and the SMOD Random Forest. The next notebook can then load three frozen six-city models, built surface, population, and Degree of Urbanisation, and evaluate all nine Jaipur transitions. Jaipur must remain excluded from training.